In [ ]:
import json
import matplotlib.pyplot as plt

log_path = f'Deformable-DETR/exp14/r50_deformable_detr_plus_iterative_bbox_refinement_plus_plus_two_stage/log.txt'
epochs = []
train_loss = []
val_loss = []
val_map = []

with open(log_path, "r") as f:
    for i, line in enumerate(f):
        line = line.strip()
        if not line:
            continue
        
        data = json.loads(line)

        # train loss
        if "train_loss" in data:
            train_loss.append(data["train_loss"])
        else:
            train_loss.append(None)

        # val loss
        if "test_loss" in data:
            val_loss.append(data["test_loss"])
        else:
            val_loss.append(None)

        # val mAP
        if "test_coco_eval_bbox" in data:
            val_map.append(data["test_coco_eval_bbox"][0])
        else:
            val_map.append(None)

        epochs.append(i)

plt.figure()
plt.plot(epochs, train_loss, label="Train Loss")
plt.plot(epochs, val_loss, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Val Loss")
plt.legend()
plt.grid()
plt.show()

plt.figure()
plt.plot(epochs, val_map, label="Val mAP")
plt.xlabel("Epoch")
plt.ylabel("mAP (AP@[0.5:0.95])")
plt.title("Validation mAP of exp7")
plt.legend()
plt.grid()
plt.show()


epochs = []
maps = []

with open(log_path, "r") as f:
    for line in f:
        data = json.loads(line)
        if "test_coco_eval_bbox" in data:
            epochs.append(data["epoch"])
            maps.append(data["test_coco_eval_bbox"][0])

# find best
best_idx = maps.index(max(maps))
best_epoch = epochs[best_idx]
best_map = maps[best_idx]

print(f"Best epoch: {best_epoch}, mAP: {best_map:.4f}")

# plot
plt.plot(epochs, maps, marker='o')
plt.scatter(best_epoch, best_map)  # highlight
plt.ztext(best_epoch, best_map, f"  best={best_map:.3f}")

plt.xlabel("Epoch")
plt.ylabel("mAP")
plt.title(f"Validation mAP of exp8")
plt.grid()
plt.show()

In [ ]:
!pip install ensemble-boxes

In [ ]:
import json
from ensemble_boxes import weighted_boxes_fusion

def load_coco(path):
    with open(path) as f:
        data = json.load(f)

    results = {}
    for d in data:
        img_id = d["image_id"]
        if img_id not in results:
            results[img_id] = {"boxes": [], "scores": [], "labels": []}

        x, y, w, h = d["bbox"]
        box = [x/512, y/512, (x+w)/512, (y+h)/512]

        results[img_id]["boxes"].append(box)
        results[img_id]["scores"].append(d["score"])
        results[img_id]["labels"].append(d["category_id"])

    return results

paths = ["/Deformable-DETR/outputs/pred_7_13.json",
         "/Deformable-DETR/outputs/pred_8_12.json",
         "/Deformable-DETR/outputs/pred_exp13_15.json",
         "/Deformable-DETR/outputs/pred_8_10.json",
         "/Deformable-DETR/outputs/pred_6_12.json"]

all_preds = [load_coco(p) for p in paths]

final = []

for img_id in all_preds[0].keys():

    boxes_list = [p[img_id]["boxes"] for p in all_preds]
    scores_list = [p[img_id]["scores"] for p in all_preds]
    labels_list = [p[img_id]["labels"] for p in all_preds]

    boxes, scores, labels = weighted_boxes_fusion(
        boxes_list,
        scores_list,
        labels_list,
        iou_thr=0.5,
        skip_box_thr=0.05
    )

    for b, s, l in zip(boxes, scores, labels):
        x1, y1, x2, y2 = b
        final.append({
            "image_id": img_id,
            "category_id": int(l),
            "bbox": [
                x1*512,
                y1*512,
                (x2-x1)*512,
                (y2-y1)*512
            ],
            "score": float(s)
        })

with open("ensemble.json", "w") as f:
    json.dump(final, f)


In [ ]:
import json
from pathlib import Path
from collections import defaultdict
from PIL import Image
from ensemble_boxes import weighted_boxes_fusion


def build_image_size_map(img_folder, pred_path=None):
    """Build image size map from test folder using predictions as guide"""
    sizes = {}
    
    # Get image_ids from first prediction file
    if pred_path and os.path.exists(pred_path):
        with open(pred_path) as f:
            preds = json.load(f)
        img_ids_from_preds = set(d["image_id"] for d in preds)
        print(f"Found {len(img_ids_from_preds)} image_ids in predictions")
    else:
        img_ids_from_preds = None

    # Load actual image sizes from test folder
    exts = [".jpg", ".jpeg", ".png", ".bmp"]
    img_files = list(Path(img_folder).glob(f"*"))
    
    for img_file in img_files:
        if img_file.suffix.lower() not in exts:
            continue
        
        try:
            img = Image.open(img_file)
            w, h = img.size
            
            # Use filename as image_id (UUID or number)
            # If it's a number, convert to int; if UUID, keep as string
            try:
                img_id = int(img_file.stem)
            except:
                img_id = img_file.stem  # Use UUID string directly
            
            sizes[img_id] = (w, h)
        except Exception as e:
            print(f"Error reading {img_file.name}: {e}")
    
    print(f"Loaded {len(sizes)} image sizes from {img_folder}")
    if sizes:
        first_size = list(sizes.values())[0]
        print(f"   Sample size: {first_size[0]} x {first_size[1]}")
    
    return sizes



def load_coco(path, image_sizes, score_power=1.0):
    """Load predictions and normalize per image size"""
    with open(path) as f:
        data = json.load(f)

    results = defaultdict(lambda: {"boxes": [], "scores": [], "labels": []})
    missing_sizes = 0

    for d in data:
        img_id = d["image_id"]

        if img_id not in image_sizes:
            missing_sizes += 1
            continue

        w_img, h_img = image_sizes[img_id]

        x, y, w, h = d["bbox"]

        # normalize correctly
        box = [
            x / w_img,
            y / h_img,
            (x + w) / w_img,
            (y + h) / h_img
        ]

        score = d["score"] ** score_power

        results[img_id]["boxes"].append(box)
        results[img_id]["scores"].append(score)
        results[img_id]["labels"].append(d["category_id"])

    if missing_sizes > 0:
        print(f"{missing_sizes} predictions had missing image sizes")

    return results


In [ ]:

import os

TEST_IMG_FOLDER = "data/nycu-hw2-data/test"

paths = ["/Deformable-DETR/outputs/pred_7_13.json",
         "/Deformable-DETR/outputs/pred_8_12.json",
         "/Deformable-DETR/outputs/pred_exp13_15.json",
         "/Deformable-DETR/outputs/pred_8_10.json",
         "/Deformable-DETR/outputs/pred_6_12.json"]



weights = [1.0, 1.0, 1.0, 1.0, 1.0]

iou_thr = 0.6
skip_box_thr = 0.1

score_power = 1.0


# LOAD IMAGE SIZES (for EACH image in test)
print("Building image size map...")
image_sizes = build_image_size_map(TEST_IMG_FOLDER, paths[0])
print()



# LOAD ALL MODELS
print("Loading predictions...")
all_preds = [
    load_coco(p, image_sizes, score_power)
    for p in paths
]
print()

# WBF ENSEMBLE

print("Running WBF ensemble...")
final = []

all_img_ids = set()
for p in all_preds:
    all_img_ids.update(p.keys())

print(f"Processing {len(all_img_ids)} images...")

for img_id in all_img_ids:
    boxes_list = []
    scores_list = []
    labels_list = []

    for p in all_preds:
        if img_id in p:
            boxes_list.append(p[img_id]["boxes"])
            scores_list.append(p[img_id]["scores"])
            labels_list.append(p[img_id]["labels"])
        else:
            boxes_list.append([])
            scores_list.append([])
            labels_list.append([])

    boxes, scores, labels = weighted_boxes_fusion(
        boxes_list,
        scores_list,
        labels_list,
        weights=weights,
        iou_thr=iou_thr,
        skip_box_thr=skip_box_thr
    )

    w_img, h_img = image_sizes[img_id]

    for b, s, l in zip(boxes, scores, labels):
        x1, y1, x2, y2 = b

        final.append({
            "image_id": img_id,
            "category_id": int(l),
            "bbox": [
                x1 * w_img,
                y1 * h_img,
                (x2 - x1) * w_img,
                (y2 - y1) * h_img
            ],
            "score": float(s)
        })



# SAVE
with open("ensemble.json", "w") as f:
    json.dump(final, f)

print(f"\nEnsemble done! Saved {len(final)} predictions to ensemble.json")